# Aula 2 — Comparação de sequências, alinhamento e BLAST

**Disciplina:** EQM — Bioinformática e Biologia Molecular  
**Projeto didático:** continuidade da sequência recuperada na Aula 1  
**Query:** `MK270576.1` — *Grammostola pulchripes*, gene mitocondrial COI

---

## Onde estamos no curso?

Na Aula 1 aprendemos a localizar um registro no NCBI/GenBank, interpretar seus principais campos e recuperar a sequência em formato FASTA.

O ponto de partida desta aula é exatamente esse arquivo:

`01_bancos/MK270576.1.fasta`

Agora surge uma pergunta central em bioinformática:

> **Depois de obter uma sequência, como podemos descobrir com quais outras sequências ela se parece e o que essa semelhança pode significar biologicamente?**

A resposta exige três ideias fundamentais:

1. **comparação de sequências**;
2. **alinhamento**;
3. **interpretação quantitativa da similaridade**.

Nesta aula utilizaremos o BLAST para transformar uma sequência isolada em uma comparação biologicamente interpretável.

## 1. Por que comparar sequências?

Uma sequência de DNA contém informação, mas essa informação raramente é interpretada de forma isolada.

Ao comparar uma sequência com outras já conhecidas podemos investigar, por exemplo:

- se há sequências muito semelhantes em outros registros;
- se uma sequência provavelmente corresponde ao mesmo gene;
- se diferentes organismos possuem versões relacionadas daquele gene;
- quais regiões são mais conservadas ou mais variáveis;
- se uma sequência desconhecida apresenta evidências de proximidade com sequências previamente identificadas;
- se uma região curta está contida dentro de uma sequência maior.

No nosso exemplo, a query corresponde a uma região de **COI — cytochrome c oxidase subunit I**.

O COI é um gene mitocondrial amplamente utilizado em estudos de diversidade, sistemática molecular e identificação baseada em DNA. Isso não significa que um resultado de BLAST, sozinho, determine automaticamente uma espécie. O BLAST fornece **evidência de similaridade de sequência**, que deve ser interpretada no contexto dos registros comparados.

## 2. A ideia de alinhamento

Considere duas sequências pequenas:

```text
Sequência A
ATGCTAGCTACGTA

Sequência B
ATGCTAGTTACGTA
```

Quando as colocamos em correspondência:

```text
ATGCTAGCTACGTA
||||||| ||||||
ATGCTAGTTACGTA
```

podemos identificar posições iguais e diferentes.

Um alinhamento pode conter:

- **match** — bases correspondentes iguais;
- **mismatch** — bases diferentes em posições alinhadas;
- **gap** — espaço introduzido para representar uma possível inserção ou deleção na comparação.

Exemplo:

```text
Query    ATGCTAGCTACGTAGGCTAA
         ||||||| ||||||| ||||
Subject  ATGCTAG-TACGTAGACTAA
                ↑       ↑
               gap   mismatch
```

O alinhamento é a estrutura que permite calcular medidas como identidade e pontuação.

## 3. Alinhamento global e alinhamento local

Nem toda comparação precisa envolver as sequências inteiras.

### Alinhamento global

Procura alinhar as sequências de uma extremidade à outra. É mais apropriado quando queremos comparar sequências de comprimento semelhante e correspondência ao longo de praticamente toda a extensão.

### Alinhamento local

Procura **regiões de alta similaridade** dentro de sequências que podem ter comprimentos diferentes.

Isso é particularmente importante quando:

- nossa query representa apenas parte de um gene;
- a sequência do banco contém uma região muito maior;
- queremos descobrir um trecho conservado dentro de uma sequência extensa.

O BLAST é fundamentalmente uma ferramenta de **busca por similaridade local**.

```text
QUERY
        ─────────────────────

SEQUÊNCIA DO BANCO
────────────────████████████────────────
                ↑
        região local semelhante
```

Nesta aula isso ficará evidente porque nosso banco didático contém tanto sequências parciais de COI quanto uma sequência mitocondrial muito maior.

## 4. Similaridade, identidade e homologia não são sinônimos

Esses termos devem ser usados com cuidado.

### Identidade

É uma medida diretamente calculável no alinhamento.

Se 98 de 100 posições alinhadas são iguais:

```text
identidade = 98%
```

### Similaridade

É um conceito mais amplo usado para descrever o grau de semelhança entre sequências. Em nucleotídeos, a porcentagem de identidade costuma ser uma das medidas mais diretamente observadas.

### Homologia

Homologia é uma relação de ancestralidade comum.

Por isso, frases como:

> “as sequências são 80% homólogas”

não são uma boa formulação.

O mais adequado seria dizer:

> “as sequências apresentam 80% de identidade nesse alinhamento”

e então discutir se o conjunto das evidências é compatível com uma relação homóloga.

Nesta prática, portanto, vamos medir **similaridade/identidade de sequência** e discutir cuidadosamente o que pode ou não ser inferido a partir disso.

## 5. O que é BLAST?

BLAST significa:

**Basic Local Alignment Search Tool**

De forma simplificada, o BLAST recebe uma sequência de consulta — a **query** — e procura regiões semelhantes em sequências de um banco de dados.

Vocabulário importante:

| Termo | Significado |
|---|---|
| **query** | sequência que estamos investigando |
| **database** | conjunto de sequências usado para a busca |
| **subject** | sequência do banco que está sendo comparada com a query |
| **hit** | resultado encontrado pelo BLAST |
| **alignment** | correspondência entre uma região da query e uma região do subject |
| **HSP** | *High-scoring Segment Pair*, segmento local de alta pontuação |

O BLAST não realiza uma busca exaustiva de todas as possibilidades como um algoritmo global completo. Ele usa uma estratégia heurística para encontrar rapidamente regiões promissoras de similaridade e estendê-las.

## 6. Como vamos interpretar um resultado?

Nesta prática não vamos aceitar a regra simplista:

```text
primeiro hit = resposta correta
```

Vamos observar conjuntamente pelo menos quatro medidas.

### 6.1 Identidade (`pident`)

Percentual de posições iguais dentro da região alinhada.

### 6.2 Cobertura da query (`qcovs`)

Quanto da nossa query foi representado no alinhamento.

Compare:

```text
Hit A
Identidade = 100%
Cobertura  = 12%
```

com:

```text
Hit B
Identidade = 98,7%
Cobertura  = 100%
```

O primeiro valor de identidade, isoladamente, não torna o Hit A necessariamente mais informativo.

### 6.3 E-value

Estima quantas correspondências com pontuação igual ou melhor seriam esperadas ao acaso, considerando o tamanho da busca.

Em geral, quanto menor o E-value, mais forte é a evidência estatística de que o alinhamento não surgiu casualmente.

Valores extremamente pequenos podem aparecer em notação científica:

```text
2e-50
7e-120
```

e alinhamentos muito fortes podem aparecer como:

```text
0.0
```

por limitação de representação numérica.

### 6.4 Bitscore

É uma pontuação normalizada da qualidade do alinhamento.

Em uma mesma busca, valores maiores de bitscore indicam alinhamentos mais bem pontuados.

### Interpretação conjunta

```text
identidade
    +
cobertura
    +
E-value
    +
bitscore
    +
descrição e procedência do registro
    ↓
interpretação
```

## 7. Por que NÃO usaremos `blastn -remote -db nt` como atividade principal?

O banco `nt` do NCBI é enorme e a busca remota depende:

- da conexão com a internet;
- do serviço do NCBI;
- da fila de processamento;
- da carga do servidor;
- de limitações de rede do ambiente de execução.

Isso torna o tempo imprevisível para uma aula.

Além disso, uma prática inicial de BLAST não precisa de milhões de sequências para ensinar os conceitos fundamentais.

Nesta aula construiremos um **banco local didático pequeno**, contendo sequências reais obtidas do NCBI. A busca local será rápida e reproduzível.

O banco incluirá exemplos em diferentes níveis de proximidade, incluindo:

- outro registro de *Grammostola pulchripes*;
- uma tarântula de outro gênero;
- outras aranhas;
- um artrópode não-aranha como comparação externa.

A consulta ao NCBI será usada apenas para recuperar poucos registros FASTA. A busca BLAST propriamente dita ocorrerá **localmente no Colab**.

## 8. Fluxo completo da aula

```text
Aula 1
GenBank
   ↓
MK270576.1.fasta
   ↓
Aula 2
entender alinhamento
   ↓
preparar ambiente Conda
   ↓
instalar BLAST+
   ↓
obter pequeno conjunto de referências
   ↓
banco_coi_didatico.fasta
   ↓
makeblastdb
   ↓
banco BLAST local
   ↓
blastn
   ↓
blast_resultado.tsv
   ↓
identidade + cobertura + E-value + bitscore
   ↓
interpretação biológica
```

### Entrada principal

`01_bancos/MK270576.1.fasta`

### Saídas principais

`02_blast/banco_coi_didatico.fasta`

`02_blast/blast_resultado.tsv`

`02_blast/blast_alinhamentos.txt`

Esses arquivos permanecerão no Google Drive.

## 9. Runtime, ambiente e persistência

O Google Colab fornece uma máquina temporária.

Isso significa que devemos distinguir:

```text
RUNTIME DO COLAB
Conda
BLAST+
programas instalados
        ↓
podem desaparecer quando a sessão termina
```

de:

```text
GOOGLE DRIVE
FASTA
resultados
relatórios
        ↓
permanecem entre as aulas
```

Vamos utilizar um ambiente Conda chamado:

`bioinfo`

A ideia não é instalar todas as ferramentas do curso de uma vez. Instalaremos cada ferramenta quando surgir a necessidade analítica.

# Parte prática

## 10. Preparar o runtime e o Conda

As células marcadas como **infraestrutura** já estão preenchidas. Elas preparam o ambiente para que o tempo de aula seja dedicado à análise, e não à digitação de rotinas administrativas.

In [ ]:
import shutil

if shutil.which("conda"):
    print("Conda já está disponível neste runtime.")
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
!conda --version
!conda config --remove-key channels 2>/dev/null || true
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict

## 11. Montar o Google Drive e localizar o projeto

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
AULA1 = ROOT / "01_bancos"
AULA2 = ROOT / "02_blast"
AMBIENTES = ROOT / "ambientes"

AULA2.mkdir(parents=True, exist_ok=True)
AMBIENTES.mkdir(parents=True, exist_ok=True)

QUERY = AULA1 / "MK270576.1.fasta"
BANCO_FASTA = AULA2 / "banco_coi_didatico.fasta"
DB_PREFIX = AULA2 / "banco_coi"
RESULTADO = AULA2 / "blast_resultado.tsv"
ALINHAMENTOS = AULA2 / "blast_alinhamentos.txt"

os.chdir(ROOT)

print("Query:", QUERY)
print("Diretório da Aula 2:", AULA2)

## 12. Verificar a entrada da Aula 1

### Tarefa

Confirme que `MK270576.1.fasta` existe e mostre as primeiras linhas do arquivo.

**Antes de programar, responda:** o que representa a linha iniciada por `>` em um FASTA?

In [ ]:
# SUA VEZ
# 1. Verifique se QUERY existe.
# 2. Se não existir, interrompa a análise com uma mensagem clara.
# 3. Mostre o início do FASTA.

## 13. Preparar o ambiente `bioinfo` com BLAST+

### Tarefa

No ambiente `bioinfo`:

1. crie o ambiente se necessário;
2. instale BLAST+;
3. verifique `blastn`;
4. verifique `makeblastdb`.

Não instale programas que ainda não serão utilizados nesta aula.

In [ ]:
# SUA VEZ — ambiente e BLAST+

## 14. Preparar o pequeno conjunto de referências

Esta é uma célula de **infraestrutura**, pois o objetivo da aula não é escrever uma requisição HTTP.

O conjunto inclui registros reais do NCBI em diferentes níveis de proximidade.

In [ ]:
import urllib.parse
import urllib.request

ACCESSIONS = [
    "MG273517.1",
    "NC_053738.1",
    "KT383764.1",
    "MK936297.1",
    "OL874968.1",
    "MT072821.1",
]

if BANCO_FASTA.exists() and BANCO_FASTA.stat().st_size > 0:
    print("Banco já disponível:", BANCO_FASTA)
else:
    ids = ",".join(ACCESSIONS)
    params = urllib.parse.urlencode({
        "db": "nuccore",
        "id": ids,
        "rettype": "fasta",
        "retmode": "text",
    })
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?" + params

    with urllib.request.urlopen(url, timeout=60) as response:
        fasta = response.read().decode("utf-8")

    if not fasta.lstrip().startswith(">"):
        raise RuntimeError("O NCBI não retornou um FASTA válido.")

    BANCO_FASTA.write_text(fasta)

print("Registros no banco:", BANCO_FASTA.read_text().count(">"))
print(BANCO_FASTA.read_text()[:1000])

## 15. Criar o banco BLAST

### Antes de escrever o comando

Explique com suas palavras:

**Qual é a diferença entre `banco_coi_didatico.fasta` e o banco indexado que será criado por `makeblastdb`?**

### Tarefa

Use `makeblastdb` para transformar o FASTA em um banco de nucleotídeos.

In [ ]:
# SUA VEZ — makeblastdb

## 16. Executar o BLAST local

### Tarefa

Compare `MK270576.1.fasta` com o banco local usando `blastn`.

A saída deverá incluir, no mínimo:

- identificador da query;
- identificador do subject;
- porcentagem de identidade;
- comprimento do alinhamento;
- cobertura da query;
- E-value;
- bitscore;
- descrição do registro.

Salve a tabela em `blast_resultado.tsv`.

In [ ]:
# SUA VEZ — blastn local

## 17. Abrir a saída tabular com pandas

### Tarefa

Leia `blast_resultado.tsv`, atribua nomes às colunas e visualize a tabela.

In [ ]:
# SUA VEZ — leitura da tabela

## 18. Interpretar os hits

Crie uma tabela contendo pelo menos:

- subject;
- identidade;
- cobertura;
- E-value;
- bitscore;
- descrição.

Depois responda às perguntas abaixo.

In [ ]:
# SUA VEZ — selecione e organize as métricas importantes

### Questões de interpretação

1. Qual hit possui maior bitscore?
2. O hit com maior identidade possui também alta cobertura?
3. O que acontece quando a query é comparada ao mitogenoma de uma tarântula?
4. Resultados de outras aranhas ainda são detectados?
5. O registro de inseto se comporta de maneira diferente?
6. Por que `100% de identidade` não deve ser interpretado sem observar cobertura?
7. O primeiro hit permite, sozinho, afirmar uma identificação taxonômica?

## 19. Visualizar o alinhamento

### Tarefa

Execute novamente `blastn`, desta vez usando um formato de saída textual que mostre os alinhamentos.

Salve em `blast_alinhamentos.txt`.

In [ ]:
# SUA VEZ — saída textual do alinhamento

### Observe no alinhamento

- início e fim da Query;
- início e fim do Subject;
- matches;
- mismatches;
- gaps;
- extensão da região comparada.

Relacione o que você vê com a porcentagem de identidade apresentada anteriormente.

## 20. Registrar o ambiente

### Tarefa

Exporte o histórico do ambiente `bioinfo` para:

`ambientes/aula02_bioinfo.yml`

In [ ]:
# SUA VEZ — exporte o ambiente

# Fechamento da Aula 2

Nesta aula partimos de uma única sequência recuperada do GenBank e realizamos a primeira análise comparativa da disciplina.

O caminho foi:

```text
sequência FASTA
      ↓
banco de referências
      ↓
alinhamento local
      ↓
BLAST
      ↓
hits
      ↓
identidade
cobertura
E-value
bitscore
      ↓
interpretação
```

## O que deve permanecer no Google Drive?

```text
01_bancos/
└── MK270576.1.fasta

02_blast/
├── banco_coi_didatico.fasta
├── banco_coi.*
├── blast_resultado.tsv
└── blast_alinhamentos.txt

ambientes/
└── aula02_bioinfo.yml
```

## Conexão com a próxima aula

A Aula 2 trabalha com uma sequência pronta e pequena.

Na próxima etapa mudaremos de escala: entraremos em **dados brutos de sequenciamento**.

```text
uma sequência pronta
        ↓
      BLAST

depois:

SRA
 ↓
milhares de reads brutos
 ↓
FASTQ
 ↓
controle de qualidade
 ↓
trimming
 ↓
montagem
 ↓
contigs
 ↓
UCEs
```

O resultado do BLAST não é uma entrada computacional obrigatória para o pipeline de SRA. A conexão é conceitual: primeiro aprendemos a pensar sobre comparação de sequências; depois passamos a trabalhar com conjuntos muito maiores de dados.